> **中文备注（自用）**
>
> **本文件 = 检索调参 / 消融实验版**，用来遍历多种检索配置、搜最优超参。
>
> - **只优化检索**（看 Evidence F-score），分类标签写死为 `SUPPORTS`（majority label），所以 Accuracy 恒为 0.4416，是占位的假值。
> - 方法 = **Dual** 双路 TF-IDF 召回 + **Score Fusion** 语义分/词面分融合 + **Gap** 基于分差的自适应证据选择（对应 pipeline 的 Stage 1–3，不含分类）。
> - 网格搜索约 1127 个配置，最优：`alpha=0.03, gap=0.03`。
> - ⚠️ **本文件生成的 submission 不能直接提交**（label 全是 SUPPORTS，分类分很低）。真正提交用 `final_retrieve_classify.ipynb`（含 DeBERTa 真分类，Accuracy≈0.5649）。
> - 流程：本文件搜出最优检索参数 → 用到 `final` 完整系统 → `final` 出最终结果并提交。

# 1. Dataset Processing

Loads the climate fact-checking corpus (1,228 train / 154 dev / 153 test claims and a 1.2M-passage evidence corpus) and builds the **dual TF-IDF candidate retrieval** stage.

Two complementary TF-IDF retrievers — a precise unigram retriever (stopwords removed) and a richer unigram+bigram retriever (sublinear TF, no stopword removal) — each return top-250 candidates. They are fused via **Reciprocal Rank Fusion (RRF)** into a pool of up to 400 candidates per claim, with lexical features cached for later score fusion.

In [1]:
# ============================================================
# 1. DataSet Processing
# ============================================================

!pip install -q sentence-transformers scikit-learn tqdm gdown

import os
import json
import time
import random
import subprocess
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [2]:
# ----------------------------
# File paths
# ----------------------------
from pathlib import Path

if "COLAB_GPU" in os.environ:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DATA_DIR = Path("/content/drive/MyDrive/climate-claim-verification/data")
else:
    # Local usage: notebook is inside notebooks/
    DATA_DIR = Path("../data")

TRAIN_FILE = DATA_DIR / "train-claims.json"
DEV_FILE = DATA_DIR / "dev-claims.json"
TEST_FILE = DATA_DIR / "test-claims-unlabelled.json"
EVIDENCE_FILE = DATA_DIR / "evidence.json"
EVAL_SCRIPT = DATA_DIR / "eval.py"

# EVIDENCE_GDRIVE_ID = "1JlUzRufknsHzKzvrEjgw8D3n_IRpjzo6"

# ----------------------------
# Reproducibility
# ----------------------------
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ----------------------------
# Download evidence.json if needed (data now loaded from Google Drive)
# ----------------------------
# if not EVIDENCE_FILE.exists():
#     print("evidence.json not found. Downloading from Google Drive...")
#     !gdown --id 1JlUzRufknsHzKzvrEjgw8D3n_IRpjzo6 -O evidence.json
# else:
#     print("evidence.json already exists.")

# ----------------------------
# Load data
# ----------------------------
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("\n--- Loading data ---")

train_claims = load_json(TRAIN_FILE)
dev_claims = load_json(DEV_FILE)
test_claims = load_json(TEST_FILE)
evidence_dict = load_json(EVIDENCE_FILE)

evidence_ids = list(evidence_dict.keys())
evidence_texts = list(evidence_dict.values())

print(f"Train claims: {len(train_claims)}")
print(f"Dev claims: {len(dev_claims)}")
print(f"Test claims: {len(test_claims)}")
print(f"Evidence documents: {len(evidence_dict)}")

Mounted at /content/drive
Using device: cuda

--- Loading data ---
Train claims: 1228
Dev claims: 154
Test claims: 153
Evidence documents: 1208827


In [3]:
# ----------------------------
# Current strong retrieval config
# ----------------------------
TOP_K_A = 250
TOP_K_B = 250
FINAL_CANDIDATE_K = 400

# Fine search around the currently best alpha=0.05
ALPHA_LIST = [0.00, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08]

# Fixed top-k candidates to compare with adaptive selection
FIXED_TOP_K_LIST = [2, 3, 4, 5, 6]

# Adaptive evidence selection search
# Strategy 1: keep all evidence with score close to top score
GAP_LIST = [0.03, 0.05, 0.08, 0.10, 0.12, 0.15, 0.20]

# Strategy 2: keep all evidence above absolute normalised score threshold
THRESHOLD_LIST = [0.55, 0.60, 0.65, 0.70, 0.75, 0.80]

MIN_K_LIST = [1, 2, 3]
MAX_K_LIST = [3, 4, 5, 6]

# CrossEncoder training settings
NEGATIVE_PER_POSITIVE = 4
MAX_TRAIN_CLAIMS = 3000

BASE_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CE_MAX_LENGTH = 512

TRAIN_BATCH_SIZE = 16
PREDICT_BATCH_SIZE = 64

EPOCHS = 3
LEARNING_RATE = 1e-5
WARMUP_RATIO = 0.1

In [4]:
# ----------------------------
# TF-IDF Retriever A: stable unigram retriever
# ----------------------------
print("\n--- Building TF-IDF retriever A: unigram + stop_words='english' ---")
start_time = time.time()

vectorizer_a = TfidfVectorizer(
    token_pattern=r"(?u)\b[a-z0-9]+\b",
    lowercase=True,
    stop_words="english"
)

tfidf_matrix_a = vectorizer_a.fit_transform(evidence_texts)

print("TF-IDF A matrix shape:", tfidf_matrix_a.shape)
print(f"TF-IDF A build time: {time.time() - start_time:.2f}s")

# ----------------------------
# TF-IDF Retriever B: richer unigram+bigram retriever
# ----------------------------
print("\n--- Building TF-IDF retriever B: unigram+bigram, no stopword removal ---")
start_time = time.time()

vectorizer_b = TfidfVectorizer(
    token_pattern=r"(?u)\b[a-z0-9]+\b",
    lowercase=True,
    stop_words=None,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    max_features=500000
)

tfidf_matrix_b = vectorizer_b.fit_transform(evidence_texts)

print("TF-IDF B matrix shape:", tfidf_matrix_b.shape)
print(f"TF-IDF B build time: {time.time() - start_time:.2f}s")


--- Building TF-IDF retriever A: unigram + stop_words='english' ---
TF-IDF A matrix shape: (1208827, 531456)
TF-IDF A build time: 32.83s

--- Building TF-IDF retriever B: unigram+bigram, no stopword removal ---
TF-IDF B matrix shape: (1208827, 500000)
TF-IDF B build time: 78.47s


In [5]:
# ----------------------------
# Utility functions
# ----------------------------
def get_top_indices(scores, top_k):
    if top_k >= len(scores):
        return np.argsort(scores)[::-1]

    top_indices = np.argpartition(scores, -top_k)[-top_k:]
    top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
    return top_indices

def minmax_normalise(values):
    values = np.asarray(values, dtype=float)

    if len(values) == 0:
        return values

    v_min = float(np.min(values))
    v_max = float(np.max(values))

    if abs(v_max - v_min) < 1e-12:
        return np.zeros_like(values)

    return (values - v_min) / (v_max - v_min)

# ----------------------------
# Retriever A with scores
# ----------------------------
def retrieve_a_with_scores(claim_text, top_k=TOP_K_A):
    claim_vec = vectorizer_a.transform([claim_text])
    scores = linear_kernel(claim_vec, tfidf_matrix_a).flatten()

    top_indices = get_top_indices(scores, top_k)
    raw_scores = scores[top_indices]
    norm_scores = minmax_normalise(raw_scores)

    results = []

    for rank, (idx, raw_score, norm_score) in enumerate(
        zip(top_indices, raw_scores, norm_scores),
        start=1
    ):
        results.append({
            "evidence_id": evidence_ids[idx],
            "rank_a": rank,
            "score_a": float(raw_score),
            "norm_score_a": float(norm_score)
        })

    return results

# ----------------------------
# Retriever B with scores
# ----------------------------
def retrieve_b_with_scores(claim_text, top_k=TOP_K_B):
    claim_vec = vectorizer_b.transform([claim_text])
    scores = linear_kernel(claim_vec, tfidf_matrix_b).flatten()

    top_indices = get_top_indices(scores, top_k)
    raw_scores = scores[top_indices]
    norm_scores = minmax_normalise(raw_scores)

    results = []

    for rank, (idx, raw_score, norm_score) in enumerate(
        zip(top_indices, raw_scores, norm_scores),
        start=1
    ):
        results.append({
            "evidence_id": evidence_ids[idx],
            "rank_b": rank,
            "score_b": float(raw_score),
            "norm_score_b": float(norm_score)
        })

    return results

# ----------------------------
# Ensemble candidate retrieval with lexical features
# ----------------------------
def retrieve_candidates_ensemble_with_features(
    claim_text,
    top_k_a=TOP_K_A,
    top_k_b=TOP_K_B,
    final_k=FINAL_CANDIDATE_K
):
    results_a = retrieve_a_with_scores(claim_text, top_k_a)
    results_b = retrieve_b_with_scores(claim_text, top_k_b)

    candidate_info = {}

    for item in results_a:
        eid = item["evidence_id"]
        candidate_info[eid] = {
            "evidence_id": eid,
            "rank_a": item["rank_a"],
            "rank_b": None,
            "score_a": item["score_a"],
            "score_b": 0.0,
            "norm_score_a": item["norm_score_a"],
            "norm_score_b": 0.0
        }

    for item in results_b:
        eid = item["evidence_id"]

        if eid not in candidate_info:
            candidate_info[eid] = {
                "evidence_id": eid,
                "rank_a": None,
                "rank_b": item["rank_b"],
                "score_a": 0.0,
                "score_b": item["score_b"],
                "norm_score_a": 0.0,
                "norm_score_b": item["norm_score_b"]
            }
        else:
            candidate_info[eid]["rank_b"] = item["rank_b"]
            candidate_info[eid]["score_b"] = item["score_b"]
            candidate_info[eid]["norm_score_b"] = item["norm_score_b"]

    # Lexical score = TF-IDF signal + light reciprocal rank fusion signal
    for eid, info in candidate_info.items():
        rrf = 0.0

        if info["rank_a"] is not None:
            rrf += 1.0 / (60.0 + info["rank_a"])

        if info["rank_b"] is not None:
            rrf += 1.0 / (60.0 + info["rank_b"])

        info["rrf_score"] = float(rrf)
        info["lexical_score"] = float(
            max(info["norm_score_a"], info["norm_score_b"]) + 5.0 * rrf
        )

    # Interleave A and B to avoid one retriever fully dominating
    merged = []
    seen = set()

    max_len = max(len(results_a), len(results_b))

    for i in range(max_len):
        if i < len(results_a):
            eid = results_a[i]["evidence_id"]
            if eid not in seen:
                seen.add(eid)
                merged.append(candidate_info[eid])

        if i < len(results_b):
            eid = results_b[i]["evidence_id"]
            if eid not in seen:
                seen.add(eid)
                merged.append(candidate_info[eid])

        if len(merged) >= final_k:
            break

    return merged[:final_k]

In [6]:
# ----------------------------
# Pre-compute dev candidates
# ----------------------------
print("\n--- Pre-computing dev ensemble candidates with lexical features ---")
start_time = time.time()

dev_candidate_cache = {}

for claim_id, claim_info in tqdm(dev_claims.items(), desc="Dev candidate retrieval"):
    claim_text = claim_info["claim_text"]

    candidate_infos = retrieve_candidates_ensemble_with_features(
        claim_text,
        top_k_a=TOP_K_A,
        top_k_b=TOP_K_B,
        final_k=FINAL_CANDIDATE_K
    )

    dev_candidate_cache[claim_id] = {
        "claim_text": claim_text,
        "candidate_infos": candidate_infos
    }

print(f"Finished dev candidate retrieval in {(time.time() - start_time):.2f}s")

candidate_lens = [len(x["candidate_infos"]) for x in dev_candidate_cache.values()]
print("\nCandidate length stats:")
print("min:", min(candidate_lens))
print("max:", max(candidate_lens))
print("mean:", float(np.mean(candidate_lens)))


--- Pre-computing dev ensemble candidates with lexical features ---


Dev candidate retrieval:   0%|          | 0/154 [00:00<?, ?it/s]

Finished dev candidate retrieval in 240.31s

Candidate length stats:
min: 295
max: 400
mean: 388.0844155844156


# 2. Model Implementation

Fine-tunes a **CrossEncoder** (`ms-marco-MiniLM-L-6-v2`) to re-rank the TF-IDF candidates by semantic claim–evidence relevance, then defines the scoring and selection machinery used in the search:

- **Training pairs** — gold evidence as positives, random non-gold evidence as negatives (4:1).
- **Score fusion** — combines the normalised CrossEncoder score with the lexical RRF score, controlled by `alpha` (alpha=0 → pure CrossEncoder).
- **Evidence selection** — three strategies compared in Section 3: fixed top-k, gap-based adaptive, and absolute-threshold adaptive (the adaptive ones pick a variable number of passages per claim).

Dev candidates are scored once and cached so the hyperparameter search reuses CrossEncoder scores.

In [7]:
# ============================================================
# 2. Model Implementation
# ============================================================

from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader

# ----------------------------
# Build CrossEncoder training pairs
# ----------------------------
print("\n--- Building random-negative CrossEncoder training pairs ---")
start_time = time.time()

train_samples = []
train_items = list(train_claims.items())

if MAX_TRAIN_CLAIMS is not None:
    train_items = train_items[:MAX_TRAIN_CLAIMS]

all_evidence_set = set(evidence_ids)

for claim_id, claim_info in tqdm(train_items, desc="Training pair construction"):
    claim_text = claim_info["claim_text"]
    gold_evidence_ids = claim_info.get("evidences", [])

    gold_set = set(gold_evidence_ids)

    # Positive examples
    for gold_id in gold_evidence_ids:
        if gold_id in evidence_dict:
            train_samples.append(
                InputExample(
                    texts=[claim_text, evidence_dict[gold_id]],
                    label=1.0
                )
            )

    # Random negatives
    n_negatives = len(gold_evidence_ids) * NEGATIVE_PER_POSITIVE

    if n_negatives <= 0:
        continue

    negative_pool = list(all_evidence_set - gold_set)

    sampled_negative_ids = random.sample(
        negative_pool,
        min(n_negatives, len(negative_pool))
    )

    for neg_id in sampled_negative_ids:
        train_samples.append(
            InputExample(
                texts=[claim_text, evidence_dict[neg_id]],
                label=0.0
            )
        )

print(f"Training samples: {len(train_samples)}")
print(f"Pair construction time: {(time.time() - start_time) / 60:.2f} min")


--- Building random-negative CrossEncoder training pairs ---


Training pair construction:   0%|          | 0/1228 [00:00<?, ?it/s]

Training samples: 20610
Pair construction time: 4.60 min


In [8]:
# ----------------------------
# Load and fine-tune CrossEncoder
# ----------------------------
print("\n--- Loading and fine-tuning CrossEncoder ---")

ce_model = CrossEncoder(
    BASE_MODEL_NAME,
    num_labels=1,
    max_length=CE_MAX_LENGTH,
    device=DEVICE
)

train_dataloader = DataLoader(
    train_samples,
    shuffle=True,
    batch_size=TRAIN_BATCH_SIZE
)

warmup_steps = int(len(train_dataloader) * EPOCHS * WARMUP_RATIO)

print("Base model:", BASE_MODEL_NAME)
print("Training batches:", len(train_dataloader))
print("Epochs:", EPOCHS)
print("Warmup steps:", warmup_steps)
print("Device:", DEVICE)

start_time = time.time()

ce_model.fit(
    train_dataloader=train_dataloader,
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": LEARNING_RATE},
    show_progress_bar=True
)

print(f"CrossEncoder fine-tuning time: {(time.time() - start_time) / 60:.2f} min")


--- Loading and fine-tuning CrossEncoder ---


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Base model: cross-encoder/ms-marco-MiniLM-L-6-v2
Training batches: 1289
Epochs: 3
Warmup steps: 386
Device: cuda


Step,Training Loss
500,0.215408
1000,0.046142
1500,0.036710
2000,0.029936
2500,0.022336
3000,0.017426
3500,0.014348


CrossEncoder fine-tuning time: 3.57 min


In [9]:
# ----------------------------
# CrossEncoder scoring
# ----------------------------
def score_claim_candidates_with_features(claim_text, candidate_infos):
    """
    Score candidates with fine-tuned CrossEncoder.
    Keep lexical scores for fusion.
    """
    candidate_ids = [item["evidence_id"] for item in candidate_infos]

    pairs = [
        [claim_text, evidence_dict.get(eid, "")]
        for eid in candidate_ids
    ]

    ce_scores = ce_model.predict(
        pairs,
        batch_size=PREDICT_BATCH_SIZE,
        show_progress_bar=False
    )

    scored_items = []

    for item, ce_score in zip(candidate_infos, ce_scores):
        new_item = dict(item)
        new_item["ce_score"] = float(ce_score)
        scored_items.append(new_item)

    scored_items = sorted(
        scored_items,
        key=lambda x: x["ce_score"],
        reverse=True
    )

    return scored_items

In [10]:
# ----------------------------
# Score fusion
# ----------------------------
def rank_with_score_fusion(scored_items, alpha=0.05):
    """
    alpha = 0.0 means pure CrossEncoder.
    alpha > 0 adds lexical TF-IDF/RRF signal.
    """
    if len(scored_items) == 0:
        return []

    ce_values = np.array([item["ce_score"] for item in scored_items], dtype=float)
    lex_values = np.array([item["lexical_score"] for item in scored_items], dtype=float)

    norm_ce = minmax_normalise(ce_values)
    norm_lex = minmax_normalise(lex_values)

    fused_items = []

    for item, ce_norm, lex_norm in zip(scored_items, norm_ce, norm_lex):
        new_item = dict(item)
        new_item["norm_ce_score"] = float(ce_norm)
        new_item["norm_lexical_score"] = float(lex_norm)
        new_item["final_score"] = float((1.0 - alpha) * ce_norm + alpha * lex_norm)
        fused_items.append(new_item)

    fused_items = sorted(
        fused_items,
        key=lambda x: x["final_score"],
        reverse=True
    )

    return fused_items

# ----------------------------
# Evidence selection strategies
# ----------------------------
def select_evidences_fixed(fused_items, top_k):
    return [item["evidence_id"] for item in fused_items[:top_k]]

def select_evidences_by_gap(fused_items, gap, min_k=1, max_k=5):
    """
    Select evidence whose final score is close to the top evidence score.
    This adapts evidence count per claim.
    """
    if len(fused_items) == 0:
        return []

    top_score = fused_items[0]["final_score"]

    selected = [
        item["evidence_id"]
        for item in fused_items
        if (top_score - item["final_score"]) <= gap
    ]

    if len(selected) < min_k:
        selected = [item["evidence_id"] for item in fused_items[:min_k]]

    selected = selected[:max_k]

    return selected

def select_evidences_by_threshold(fused_items, threshold, min_k=1, max_k=5):
    """
    Select evidence above an absolute final score threshold.
    Scores are normalised per claim, so threshold is between 0 and 1.
    """
    if len(fused_items) == 0:
        return []

    selected = [
        item["evidence_id"]
        for item in fused_items
        if item["final_score"] >= threshold
    ]

    if len(selected) < min_k:
        selected = [item["evidence_id"] for item in fused_items[:min_k]]

    selected = selected[:max_k]

    return selected

In [11]:
# ----------------------------
# Build predictions from scored cache
# ----------------------------
def build_predictions_from_scored_cache(scored_cache, config):
    """
    Build predictions using selected scoring and evidence selection config.
    """
    predictions = {}

    alpha = config["alpha"]
    selection_type = config["selection_type"]

    for claim_id, info in scored_cache.items():
        fused_items = rank_with_score_fusion(
            info["scored_items"],
            alpha=alpha
        )

        if selection_type == "fixed":
            evidence_ids_selected = select_evidences_fixed(
                fused_items,
                top_k=config["top_k"]
            )

        elif selection_type == "gap":
            evidence_ids_selected = select_evidences_by_gap(
                fused_items,
                gap=config["gap"],
                min_k=config["min_k"],
                max_k=config["max_k"]
            )

        elif selection_type == "threshold":
            evidence_ids_selected = select_evidences_by_threshold(
                fused_items,
                threshold=config["threshold"],
                min_k=config["min_k"],
                max_k=config["max_k"]
            )

        else:
            raise ValueError(f"Unknown selection type: {selection_type}")

        predictions[claim_id] = {
            "claim_text": info["claim_text"],
            "claim_label": "SUPPORTS",
            "evidences": evidence_ids_selected
        }

    return predictions

In [12]:
# ----------------------------
# Score dev candidates once
# ----------------------------
print("\n--- Scoring dev candidates once with CrossEncoder ---")
start_time = time.time()

dev_scored_cache = {}

for i, (claim_id, info) in enumerate(dev_candidate_cache.items(), start=1):
    claim_text = info["claim_text"]
    candidate_infos = info["candidate_infos"]

    scored_items = score_claim_candidates_with_features(
        claim_text,
        candidate_infos
    )

    dev_scored_cache[claim_id] = {
        "claim_text": claim_text,
        "scored_items": scored_items
    }

    if i % 10 == 0 or i == len(dev_candidate_cache):
        elapsed = time.time() - start_time
        print(
            f"[Progress] {i}/{len(dev_candidate_cache)} | "
            f"elapsed={elapsed / 60:.1f} min | "
            f"avg={elapsed / i:.2f}s/claim"
        )

print(f"Dev CrossEncoder scoring time: {(time.time() - start_time) / 60:.2f} min")


--- Scoring dev candidates once with CrossEncoder ---
[Progress] 10/154 | elapsed=0.0 min | avg=0.27s/claim
[Progress] 20/154 | elapsed=0.1 min | avg=0.29s/claim
[Progress] 30/154 | elapsed=0.1 min | avg=0.29s/claim
[Progress] 40/154 | elapsed=0.2 min | avg=0.28s/claim
[Progress] 50/154 | elapsed=0.2 min | avg=0.28s/claim
[Progress] 60/154 | elapsed=0.3 min | avg=0.28s/claim
[Progress] 70/154 | elapsed=0.3 min | avg=0.29s/claim
[Progress] 80/154 | elapsed=0.4 min | avg=0.29s/claim
[Progress] 90/154 | elapsed=0.4 min | avg=0.29s/claim
[Progress] 100/154 | elapsed=0.5 min | avg=0.29s/claim
[Progress] 110/154 | elapsed=0.5 min | avg=0.29s/claim
[Progress] 120/154 | elapsed=0.6 min | avg=0.29s/claim
[Progress] 130/154 | elapsed=0.6 min | avg=0.29s/claim
[Progress] 140/154 | elapsed=0.7 min | avg=0.28s/claim
[Progress] 150/154 | elapsed=0.7 min | avg=0.28s/claim
[Progress] 154/154 | elapsed=0.7 min | avg=0.28s/claim
Dev CrossEncoder scoring time: 0.73 min


# 3. Testing and Evaluation

Runs a **hyperparameter grid search** over fusion weight `alpha`, selection strategy, and the per-strategy parameters (top-k / score gap / threshold / min-max evidence count), evaluating each config on the dev set with the official `eval.py` (Evidence F-score, Accuracy, Harmonic Mean).

The best config by Harmonic Mean is then used to generate the final test predictions, followed by a sanity check (coverage, label distribution, evidence-count distribution, invalid IDs) and submission packaging.

In [13]:
# ============================================================
# 3. Testing and Evaluation
# ============================================================

# ----------------------------
# Helper functions
# ----------------------------
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

def run_eval(prediction_file, gold_file=DEV_FILE):
    command = [
        "python",
        str(EVAL_SCRIPT),
        "--predictions",
        str(prediction_file),
        "--groundtruth",
        str(gold_file)
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    print(result.stdout)

    if result.stderr:
        print(result.stderr)

    return result.stdout

def parse_eval_output(output_text):
    f_score = None
    acc = None
    hmean = None

    for line in output_text.splitlines():
        if "Evidence Retrieval F-score" in line:
            f_score = float(line.split("=")[-1].strip())
        elif "Claim Classification Accuracy" in line:
            acc = float(line.split("=")[-1].strip())
        elif "Harmonic Mean" in line:
            hmean = float(line.split("=")[-1].strip())

    return f_score, acc, hmean

In [14]:
# ----------------------------
# Build search configs
# ----------------------------
search_configs = []

# Fixed top-k configs
for alpha in ALPHA_LIST:
    for top_k in FIXED_TOP_K_LIST:
        search_configs.append({
            "selection_type": "fixed",
            "alpha": alpha,
            "top_k": top_k,
            "name": f"fixed-alpha{alpha}-k{top_k}"
        })

# Adaptive gap configs
for alpha in ALPHA_LIST:
    for gap in GAP_LIST:
        for min_k in MIN_K_LIST:
            for max_k in MAX_K_LIST:
                if min_k <= max_k:
                    search_configs.append({
                        "selection_type": "gap",
                        "alpha": alpha,
                        "gap": gap,
                        "min_k": min_k,
                        "max_k": max_k,
                        "name": f"gap-alpha{alpha}-gap{gap}-min{min_k}-max{max_k}"
                    })

# Adaptive threshold configs
for alpha in ALPHA_LIST:
    for threshold in THRESHOLD_LIST:
        for min_k in MIN_K_LIST:
            for max_k in MAX_K_LIST:
                if min_k <= max_k:
                    search_configs.append({
                        "selection_type": "threshold",
                        "alpha": alpha,
                        "threshold": threshold,
                        "min_k": min_k,
                        "max_k": max_k,
                        "name": f"thr-alpha{alpha}-thr{threshold}-min{min_k}-max{max_k}"
                    })

print("Total configs to evaluate:", len(search_configs))

Total configs to evaluate: 1127


In [15]:
# ----------------------------
# Dev evaluation
# ----------------------------
print("\n" + "=" * 70)
print("DEV EVALUATION: FIXED + ADAPTIVE EVIDENCE SELECTION")
print("=" * 70)

dev_results = []

for idx, config in enumerate(search_configs, start=1):
    print("\n" + "-" * 70)
    print(f"[{idx}/{len(search_configs)}] Evaluating {config['name']}")
    print("-" * 70)

    dev_predictions = build_predictions_from_scored_cache(
        dev_scored_cache,
        config
    )

    safe_name = (
        config["name"]
        .replace(".", "_")
        .replace("=", "")
        .replace(" ", "")
    )

    pred_file = f"dev-{safe_name}.json"
    save_json(dev_predictions, pred_file)

    output = run_eval(pred_file, DEV_FILE)
    f_score, acc, hmean = parse_eval_output(output)

    dev_results.append({
        "config": config,
        "file": pred_file,
        "F": f_score,
        "A": acc,
        "Mean": hmean
    })

# ----------------------------
# Show best results
# ----------------------------
dev_results_sorted = sorted(
    dev_results,
    key=lambda x: x["Mean"],
    reverse=True
)

print("\n" + "=" * 70)
print("TOP DEV RESULTS")
print("=" * 70)

for r in dev_results_sorted[:20]:
    c = r["config"]
    print(
        f"{c['name']} | "
        f"F={r['F']:.6f}, A={r['A']:.6f}, Mean={r['Mean']:.6f}, "
        f"file={r['file']}"
    )

best_result = dev_results_sorted[0]
best_config = best_result["config"]

print("\n" + "=" * 70)
print("BEST DEV RESULT")
print("=" * 70)
print("Best config:", best_config)
print("Best F:", best_result["F"])
print("Best A:", best_result["A"])
print("Best Mean:", best_result["Mean"])
print("Best dev prediction file:", best_result["file"])

Streaming output truncated to the last 5000 lines.
----------------------------------------------------------------------
[507/1127] Evaluating gap-alpha0.07-gap0.12-min1-max6
----------------------------------------------------------------------
Evidence Retrieval F-score (F)    = 0.20279745507018232
Claim Classification Accuracy (A) = 0.44155844155844154
Harmonic Mean of F and A          = 0.27794244975899823


----------------------------------------------------------------------
[508/1127] Evaluating gap-alpha0.07-gap0.12-min2-max3
----------------------------------------------------------------------
Evidence Retrieval F-score (F)    = 0.20066481137909709
Claim Classification Accuracy (A) = 0.44155844155844154
Harmonic Mean of F and A          = 0.27593283482929354


----------------------------------------------------------------------
[509/1127] Evaluating gap-alpha0.07-gap0.12-min2-max4
----------------------------------------------------------------------
Evidence Retrieval F-

In [16]:
# ----------------------------
# Generate test predictions with best config
# ----------------------------
print("\n" + "=" * 70)
print("GENERATING TEST PREDICTIONS")
print("=" * 70)

print("Using best config:", best_config)
print("Using ensemble candidates:")
print("TOP_K_A:", TOP_K_A)
print("TOP_K_B:", TOP_K_B)
print("FINAL_CANDIDATE_K:", FINAL_CANDIDATE_K)

start_time = time.time()

test_predictions = {}

for i, (claim_id, claim_info) in enumerate(test_claims.items(), start=1):
    claim_text = claim_info["claim_text"]

    candidate_infos = retrieve_candidates_ensemble_with_features(
        claim_text,
        top_k_a=TOP_K_A,
        top_k_b=TOP_K_B,
        final_k=FINAL_CANDIDATE_K
    )

    scored_items = score_claim_candidates_with_features(
        claim_text,
        candidate_infos
    )

    temp_cache = {
        claim_id: {
            "claim_text": claim_text,
            "scored_items": scored_items
        }
    }

    prediction_one = build_predictions_from_scored_cache(
        temp_cache,
        best_config
    )

    test_predictions[claim_id] = prediction_one[claim_id]

    if i % 10 == 0 or i == len(test_claims):
        elapsed = time.time() - start_time
        print(
            f"[Progress] {i}/{len(test_claims)} | "
            f"elapsed={elapsed / 60:.1f} min | "
            f"avg={elapsed / i:.2f}s/claim"
        )

output_file = "test-claims-predictions.json"
save_json(test_predictions, output_file)

print("\nSaved:", output_file)
print("Total test predictions:", len(test_predictions))

sample_claim_id = next(iter(test_predictions))
print("\nSample prediction:")
print(sample_claim_id, test_predictions[sample_claim_id])


GENERATING TEST PREDICTIONS
Using best config: {'selection_type': 'gap', 'alpha': 0.03, 'gap': 0.03, 'min_k': 2, 'max_k': 6, 'name': 'gap-alpha0.03-gap0.03-min2-max6'}
Using ensemble candidates:
TOP_K_A: 250
TOP_K_B: 250
FINAL_CANDIDATE_K: 400
[Progress] 10/153 | elapsed=0.3 min | avg=1.86s/claim
[Progress] 20/153 | elapsed=0.6 min | avg=1.89s/claim
[Progress] 30/153 | elapsed=0.9 min | avg=1.87s/claim
[Progress] 40/153 | elapsed=1.2 min | avg=1.86s/claim
[Progress] 50/153 | elapsed=1.5 min | avg=1.85s/claim
[Progress] 60/153 | elapsed=1.8 min | avg=1.84s/claim
[Progress] 70/153 | elapsed=2.1 min | avg=1.84s/claim
[Progress] 80/153 | elapsed=2.5 min | avg=1.84s/claim
[Progress] 90/153 | elapsed=2.8 min | avg=1.84s/claim
[Progress] 100/153 | elapsed=3.1 min | avg=1.84s/claim
[Progress] 110/153 | elapsed=3.4 min | avg=1.84s/claim
[Progress] 120/153 | elapsed=3.7 min | avg=1.83s/claim
[Progress] 130/153 | elapsed=4.0 min | avg=1.83s/claim
[Progress] 140/153 | elapsed=4.3 min | avg=1.83s/

In [17]:
# ----------------------------
# Sanity check
# ----------------------------
print("\n" + "=" * 70)
print("TEST PREDICTION SANITY CHECK")
print("=" * 70)

preds = load_json(output_file)

print("Number of test claims:", len(test_claims))
print("Number of predictions:", len(preds))

missing_claims = set(test_claims.keys()) - set(preds.keys())
extra_claims = set(preds.keys()) - set(test_claims.keys())

print("Missing predictions:", len(missing_claims))
print("Extra predictions:", len(extra_claims))

label_counts = {}
evidence_len_counts = {}
invalid_evidence_ids = []

for claim_id, pred in preds.items():
    label = pred.get("claim_label")
    evidences = pred.get("evidences", [])

    label_counts[label] = label_counts.get(label, 0) + 1
    evidence_len_counts[len(evidences)] = evidence_len_counts.get(len(evidences), 0) + 1

    for eid in evidences:
        if eid not in evidence_dict:
            invalid_evidence_ids.append((claim_id, eid))

print("\nLabel distribution:")
print(label_counts)

print("\nEvidence count distribution:")
print(evidence_len_counts)

print("\nInvalid evidence IDs:", len(invalid_evidence_ids))
if invalid_evidence_ids:
    print("First invalid examples:", invalid_evidence_ids[:5])

# ----------------------------
# Zip final prediction file
# ----------------------------
!zip -q submission.zip test-claims-predictions.json

print("\nCreated submission.zip")


TEST PREDICTION SANITY CHECK
Number of test claims: 153
Number of predictions: 153
Missing predictions: 0
Extra predictions: 0

Label distribution:
{'SUPPORTS': 153}

Evidence count distribution:
{2: 37, 6: 60, 5: 21, 3: 21, 4: 14}

Invalid evidence IDs: 0

Created submission.zip
